# USB-Kamera-Pipeline → CNN MNIST 8×8 (hls4ml) → HDMI 720p — PYNQ-Z1

**Aktuelles Design**: Roh-IP `myproject` (Vivado-Backend, `io_stream`) + Subset-Konverter (TLAST=1) + AXIS-Datenbreitenkonverter.

**PS↔PL-Schnittstellenvertrag (gemessen und validiert)**:
- Eingang: 64 × int16 = 128 Byte, Kodierung `ap_fixed<16,5>` → `round(pixel × 2^11)`
- Ausgang: 1 Paket mit 20 Byte (TLAST) = 10 × int16, Dekodierung `/ 2^11`
- **Pipeline-Versatz = 1**: Die Ausgabe der Inferenz k wird erst bei Eintreffen des Eingangs k+1 ausgegeben (Verhalten des generierten io_stream-Designs, gemessen identisch mit und ohne Wrapper) → Initialisierung erforderlich, die angezeigte Vorhersage entspricht dem vorherigen Frame (~33 ms).

**Faustregel**: Nach jedem Absturz, Timeout oder DMAIntErr → **das Overlay neu laden**, bevor neu gestartet wird (eine Ausgabe bleibt immer noch im PL im Umlauf, und die FIFOs können verwaiste Beats enthalten).

Referenzvalidierung: Keras 82,6 % · C-Sim 81,8 % · Hardware 82,9 % (N=200, IN_FRAC=11, shift=1).

In [ ]:
import time
import cv2
import numpy as np
from pynq import Overlay, allocate, MMIO, Clocks

# ─────────────────────────────────────────────
# 1. Overlay und Taktüberprüfung
# ─────────────────────────────────────────────
BIT = "/home/xilinx/jupyter_notebooks/video/mnist.bit"
ol = Overlay(BIT)

FCLK0_ATTENDU_MHZ = 125.0
fclk0 = Clocks.fclk0_mhz
print(f"FCLK0 = {fclk0:.2f} MHz")
if abs(fclk0 - FCLK0_ATTENDU_MHZ) > 1.0:
    print(f"⚠ FCLK0 devrait être {FCLK0_ATTENDU_MHZ} MHz "
          f"(pixel clock estimé : {fclk0 * 37.125 / (5 * 12.5):.2f} MHz)")


FCLK0 = 125.00 MHz


In [ ]:
# ─────────────────────────────────────────────
# 2. Videoauflösungen und HDMI-Framebuffer
# ─────────────────────────────────────────────
WIDTH, HEIGHT, BPP = 1280, 720, 3
stride = WIDTH * BPP
ROLL_X = 0   # Ausgleich des horizontalen Versatzes zwischen VDMA und VTC, falls erforderlich

fb = allocate(shape=(HEIGHT, WIDTH, BPP), dtype=np.uint8)
fb[:] = 0
fb.flush()
fb_addr = fb.physical_address


In [ ]:
# ─────────────────────────────────────────────
# 3. MMIO und Register
# ─────────────────────────────────────────────
vdma_mmio = MMIO(ol.ip_dict['axi_vdma_0']['phys_addr'], 0x100)
dma_mmio = MMIO(ol.ip_dict['axi_dma_0']['phys_addr'], 0x100)

# VDMA-Register (Kanal MM2S, Wiedergabe von DDR nach HDMI)
VDMA_MM2S_DMACR      = 0x00
VDMA_MM2S_VSIZE      = 0x50
VDMA_MM2S_HSIZE      = 0x54
VDMA_MM2S_STRIDE     = 0x58
VDMA_MM2S_ADDR_START = 0x5C

# Register AXI DMA ( Direct mode Register)
DMA_MM2S_DMACR  = 0x00
DMA_MM2S_DMASR  = 0x04
DMA_MM2S_SA     = 0x18
DMA_MM2S_LENGTH = 0x28
DMA_S2MM_DMACR  = 0x30
DMA_S2MM_DMASR  = 0x34
DMA_S2MM_DA     = 0x48
DMA_S2MM_LENGTH = 0x58

# Bits des DMASR
SR_HALTED = 0x1
SR_IDLE   = 0x2
SR_ERR    = 0x70


In [ ]:
# ─────────────────────────────────────────────
# 4. CNN-Puffer + ap_fixed<16,5>-Kodierung
#    (Roh-IP: Eingang 64 x int16, Ausgang 10 x int16)
# ─────────────────────────────────────────────
IN_FRAC, OUT_FRAC = 11, 11   # ap_fixed<16,5> -> 11 Fraktionsbits
                             # (empirisch bestätigt: Durchlauf 8/10/11 → 82,9 % bei 11)
N_OUT = 10
PIPE_SHIFT = 1               # Ausgang i = f(Eingang i-1), gemessen auf eigenem PL

in_buffer  = allocate(shape=(64,), dtype=np.int16)   # 128 Byte
out_buffer = allocate(shape=(32,), dtype=np.int16)   # 64 Byte Spielraum;
                                                     # das eigentliche Paket ist 20 Byte groß (TLAST)

def encode_input(img8_float01):
    """img8 en float [0,1] -> ap_fixed<16,5> (int16, scale 2^11)."""
    in_buffer[:] = np.round(
        np.asarray(img8_float01).flatten() * (1 << IN_FRAC)).astype(np.int16)

def decode_output():
    """10 x int16 -> probabilites float (softmax calcule dans le PL)."""
    return np.array(out_buffer[:N_OUT], dtype=np.float32) / (1 << OUT_FRAC)


In [ ]:
# ─────────────────────────────────────────────
# 5.  DMA Funktionen
# ─────────────────────────────────────────────
def dma_reset():
    print("Réinitialisation du DMA...")
    dma_mmio.write(DMA_MM2S_DMACR, 0x4)
    dma_mmio.write(DMA_S2MM_DMACR, 0x4)
    time.sleep(0.05)
    dma_mmio.write(DMA_MM2S_DMACR, 0x0)
    dma_mmio.write(DMA_S2MM_DMACR, 0x0)
    time.sleep(0.05)
    mm2s = dma_mmio.read(DMA_MM2S_DMASR)
    s2mm = dma_mmio.read(DMA_S2MM_DMASR)
    if (mm2s & SR_HALTED) and (s2mm & SR_HALTED):
        print("✓ DMA réinitialisé")
    else:
        print(f"⚠ DMA pas halted après reset "
              f"(MM2S: 0x{mm2s:08X}, S2MM: 0x{s2mm:08X})")

def dma_start():
    """RS=1 sur les deux canaux, une seule fois (PG021 : LENGTH ignoré si halted)."""
    dma_mmio.write(DMA_MM2S_DMACR, 0x1)
    dma_mmio.write(DMA_S2MM_DMACR, 0x1)
    time.sleep(0.01)
    mm2s = dma_mmio.read(DMA_MM2S_DMASR)
    s2mm = dma_mmio.read(DMA_S2MM_DMASR)
    if (mm2s & SR_HALTED) or (s2mm & SR_HALTED):
        raise RuntimeError(f"DMA toujours halted après RS=1 "
                           f"(MM2S: 0x{mm2s:08X}, S2MM: 0x{s2mm:08X})")
    print("✓ Canaux DMA démarrés (RS=1)")

def dma_prime():
    """Amorçage du pipeline (profondeur 1) : une entrée factice dont la
    sortie reste en vol ; elle sera récupérée au premier inference()."""
    in_buffer[:] = 0
    in_buffer.flush()
    dma_mmio.write(DMA_MM2S_SA, in_buffer.physical_address)
    dma_mmio.write(DMA_MM2S_LENGTH, in_buffer.nbytes)
    t = time.time() + 1.0
    while time.time() < t:
        if dma_mmio.read(DMA_MM2S_DMASR) & SR_IDLE:
            break
    else:
        raise TimeoutError("Amorçage : MM2S n'a pas consommé l'entrée factice")
    print("✓ Pipeline CNN amorcé (1 sortie en vol)")

def inference(timeout_s=2.0):
    """Envoie in_buffer (frame k), récupère la sortie de la frame k-1
    (décalage pipeline 1 du design io_stream) et retourne les 10
    probabilités décodées."""
    in_buffer.flush()

    # Den Empfang VOR dem Senden aktivieren (S2MM-Kanal befindet sich zu diesem Zeitpunkt im Leerlauf)
    dma_mmio.write(DMA_S2MM_DA, out_buffer.physical_address)
    dma_mmio.write(DMA_S2MM_LENGTH, out_buffer.nbytes)   # Sicherheitsmarge; TLAST auf 20 begrenzt

    dma_mmio.write(DMA_MM2S_SA, in_buffer.physical_address)
    dma_mmio.write(DMA_MM2S_LENGTH, in_buffer.nbytes)    # 128 Byte

    t_limit = time.time() + timeout_s
    while time.time() < t_limit:
        s = dma_mmio.read(DMA_S2MM_DMASR)
        if s & SR_ERR:
            raise RuntimeError(f"Erreur DMA S2MM: 0x{s:08X} "
                               "(recharger l'overlay avant de relancer)")
        if s & SR_IDLE:
            break
    else:
        raise TimeoutError(
            f"S2MM expiré - MM2S: 0x{dma_mmio.read(DMA_MM2S_DMASR):08X}, "
            f"S2MM: 0x{dma_mmio.read(DMA_S2MM_DMASR):08X} "
            "(recharger l'overlay avant de relancer)")

    out_buffer.invalidate()
    return decode_output()

def dma_stop():
    dma_mmio.write(DMA_MM2S_DMACR, 0x0)
    dma_mmio.write(DMA_S2MM_DMACR, 0x0)


In [ ]:
# ─────────────────────────────────────────────
# 6. VDMA (Auslesen des Framebuffers -> HDMI)
# ─────────────────────────────── ──────────────
def vdma_start():
    print(„VDMA wird gestartet...“)
    vdma_mmio.write(VDMA_MM2S_DMACR, 0x4)
    time.sleep(0.1)
    vdma_mmio.write(VDMA_MM2S_DMACR, 0x1)
    for i in range(3):
        vdma_mmio.write(VDMA_MM2S_ADDR_START + i * 4, fb_addr)
    vdma_mmio.write(VDMA_MM2S_STRIDE, stride)
    vdma_mmio.write(VDMA_MM2S_HSIZE, WIDTH * BPP)
    vdma_mmio.write(VDMA_MM2S_VSIZE, HEIGHT)   # VSIZE zuletzt: löst den Vorgang aus
    print(„✓ VDMA gestartet“)

def vdma_stop():
    vdma_mmio.write(VDMA_MM2S_DMACR, 0x4)


In [ ]:
# ─────────────────────────────────────────────
# 7. Kameravorverarbeitung nach MNIST-Art
# ──────────────────────────────────── ─────────
ROI = 300   # zentraler quadratischer Bereich in Pixeln

def preprocess_mnist(frame):
    h, w = frame.shape[:2]
    y0, x0 = (h - ROI)//2, (w - ROI)//2
    roi = frame[y0:y0+ROI, x0:x0+ROI]

    gray = cv2.cvtColor(roi, cv2.COLOR_BGR2GRAY)
    gray = cv2.GaussianBlur(gray, (5, 5), 0)

    # Otsu-Binarisierung + INVERTIERUNG (MNIST = weiße Ziffer auf schwarzem Hintergrund)
    _, binary = cv2.threshold(gray, 0, 255,
                              cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)

    # Zentrierung auf die Ziffer (wie bei MNIST)
    cnts, _ = cv2.findContours(binary, cv2.RETR_EXTERNAL,
                               cv2.CHAIN_APPROX_SIMPLE)
    if cnts:
        x, y, bw, bh = cv2.boundingRect(max(cnts, key=cv2.contourArea))
        if bw * bh > 400:          # Rauschen ignorieren
            side = int(max(bw, bh) * 1.4)   # Rand ~MNIST
            cx, cy = x + bw//2, y + bh//2
            x1 = max(cx - side//2, 0); y1 = max(cy - side//2, 0)
            binary = binary[y1:y1+side, x1:x1+side]

    small = cv2.resize(binary, (8, 8), interpolation=cv2.INTER_AREA)
    return small.astype(np.float32) / 255.0


## Validation hardware (500 images MNIST) — optionnelle
À exécuter **seule, sur PL propre** (le rechargement d'overlay est intégré). Ne pas enchaîner avec la boucle caméra sans recharger l'overlay ensuite.

In [ ]:
# ─────────────────────────────────────────────
# V. Validierung von 500 Bildern (Protokoll: Initialisierung + Versatz 1)
#    Erzielte Referenzwerte: Keras 82,6 % | C-Sim 81,8 % | HW 82,9 % (N=200)
# ─────────────────────────────────────────────
x = np.load(‚x_test_8x8.npy‘); y = np.load(‚y_test.npy‘)
assert x.shape[1:] == (8, 8, 1), f„unerwartete Form: {x.shape}“

# Eigenes PL erforderlich für eine exakte Zuordnung
ol = Overlay(BIT)
vdma_mmio = MMIO(ol.ip_dict[‚axi_vdma_0‘][‚phys_addr‘], 0x100)
dma_mmio  = MMIO(ol.ip_dict[‚axi_dma_0‘][‚phys_addr‘], 0x100)

dma_reset(); dma_start(); dma_prime ()

preds = []
for i in range(500):
    encode_input(x[i])
    probs = inference()            # Ausgabe von x[i-1] (Dummy für i=0)
    if i > 0:
        preds.append(int(np.argmax(probs)))
    if i == 1:
        print(f„Paket: {dma_mmio.read(DMA_S2MM_LENGTH)} Bytes (erwartet 20) “
              f„— Summe der Wahrscheinlichkeiten = {probs.sum():.3f} (erwartet ~1)“)

in_buffer[:] = 0; in_buffer.flush()
probs = inference ()                # leert den Ausgang von x[499]
preds.append(int(np.argmax(probs)))

preds = np.array(preds)
acc = np.mean(preds == y[:500])
print(f„Hardware-Genauigkeit: {acc*100:.1f} %  (N=500)“)

# Detailangaben pro Ziffer für die Dokumentation (Tabelle „Ergebnisse“)
for d in range(10):
    m = y[:500] == d
    if m.sum():
        print(f„Ziffer {d}: {np.mean(preds[m] == d)*100:5.1f} %  (n={m.sum()})“)


Réinitialisation du DMA...
✓ DMA réinitialisé
✓ Canaux DMA démarrés (RS=1)
✓ Pipeline CNN amorcé (1 sortie en vol)
paquet: 20 octets (attendu 20) — somme probs = 1.029 (attendu ~1)
Accuracy hardware : 82.0%  (N=500)
chiffre 0:  90.5%  (n=42)
chiffre 1: 100.0%  (n=67)
chiffre 2:  87.3%  (n=55)
chiffre 3:  77.8%  (n=45)
chiffre 4:  83.6%  (n=55)
chiffre 5:  72.0%  (n=50)
chiffre 6:  83.7%  (n=43)
chiffre 7:  81.6%  (n=49)
chiffre 8:  72.5%  (n=40)
chiffre 9:  64.8%  (n=54)


## Live-Kamera-Demo
Voraussetzung: Stromversorgung über **externe Buchse + JP5 an REG** (die USB-Resets `ci_hdrc` der C270 bei Stromversorgung über Micro-USB sind dokumentiert). Wenn der obige Test gerade erfolgreich abgelaufen ist, **das Overlay neu laden** (die Zellen 1 bis 6 erneut ausführen), bevor Sie starten.

In [ ]:
# ─────────────────────────────────────────────
# 8. USB-Kamera
# ─────────────────────────────────────── ──────
print(„Kamera wird initialisiert...“)
cap = cv2.VideoCapture(„/dev/video0“, cv2.CAP_V4L2)
time.sleep(0.5)
cap.set(cv2.CAP_PROP_FOURCC, cv2.VideoWriter_fourcc(*‚MJPG‘))
cap.set(cv2.CAP_PROP_FRAME_WIDTH, WIDTH)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, HEIGHT)

ok_count = sum(cap.read()[0] for _ in range(10))
if ok_count == 0:
    raise RuntimeError(
        „Kamera geöffnet, aber kein lesbares Bild. Überprüfen Sie:\n“
        „  - dmesg | tail (USB-Reset ‚ci_hdrc‘ = unzureichende Stromversorgung,\n“
        „    auf die externe Buchse + JP5 auf REG umschalten)\n“
        „  - fuser -v /dev/video0 (Gerät gesperrt)\n“
        „  - ls /dev/video* (die Kamera hat möglicherweise den Knoten gewechselt)“)
print(f„✓ Kamera bereit ({ok_count}/10 Warmup-Frames)“)


Initialisation de la caméra...
✓ Caméra prête (10/10 frames de warmup)


In [ ]:
# ─────────────────────────────────────────────
# 9. Starten der Hardware
# ─────────────────────────────────────────────
dma_reset()
dma_start()
dma_prime()      # füllt die Pipeline (Tiefe 1)
vdma_start()


Réinitialisation du DMA...
✓ DMA réinitialisé
✓ Canaux DMA démarrés (RS=1)
✓ Pipeline CNN amorcé (1 sortie en vol)
Démarrage du VDMA...
✓ VDMA démarré


In [ ]:
# ─────────────────────────────────────────────
# 10. Hauptschleife
# ─────────────────────────────────────────────
fail = 0
n = 0
t0 = time.time()

print(„Hauptschleife gestartet. Strg+C zum Beenden.“)
try:
    while True:
        ok, frame = cap.read()
        if not ok or frame is None:
            fail += 1
            if fail % 30 == 0:
                print(f„Kamera-Lesefehler (x{fail})“)
            time.sleep(0.01)
            continue
        fail = 0

        if frame.shape[1] != WIDTH or frame.shape[0] != HEIGHT:
            frame = cv2.resize(frame, (WIDTH, HEIGHT))

        rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

        # ── Vorverarbeitung + CNN-Inferenz ──
        img8 = preprocess_mnist(frame)
        encode_input(img8)             # ap_fixed<16,5> : round(x * 2^11)
        probs = inference()            # Wahrscheinlichkeiten des VORHERIGEN Frames

        digit = int(np.argmax(probs))
        conf = float(probs[digit])

        # ── Einblendung ──
        text = f„Vorhersage: {digit} ({conf * 100:.1f}%)“
        cv2.putText(rgb_frame, text, (50, 60), cv2.FONT_HERSHEY_SIMPLEX,
                    1.5, (0, 255, 0), 3, cv2.LINE_AA)

        # Debug-Vignette: Was das Netzwerk sieht (vergrößerte 8x8-Eingabe)
        vis = cv2.resize((img8 * 255).astype(np.uint8), (160, 160),
                         interpolation=cv2.INTER_NEAREST)
        rgb_frame[20:180, WIDTH-180:WIDTH-20] = cv2.cvtColor(vis, cv2.COLOR_GRAY2RGB)
        ry, rx = (HEIGHT-ROI)//

Boucle principale démarrée. Ctrl+C pour arrêter.
FPS: 7.3
FPS: 7.7
FPS: 7.8
FPS: 7.9
FPS: 7.9
FPS: 8.0
FPS: 8.0
FPS: 8.0
FPS: 8.1
FPS: 8.1
FPS: 8.1
FPS: 8.1
FPS: 8.1
FPS: 8.2
FPS: 8.2
FPS: 8.2
FPS: 8.2
FPS: 8.1
FPS: 8.1
FPS: 8.1
FPS: 8.1
FPS: 8.0
FPS: 8.0
FPS: 8.0
FPS: 8.0
FPS: 8.0
FPS: 8.0
FPS: 8.0
FPS: 8.0
FPS: 8.0
FPS: 8.0
FPS: 8.0
FPS: 8.0
FPS: 8.0
FPS: 8.0
FPS: 8.0
FPS: 8.1
FPS: 8.1
FPS: 8.1
FPS: 8.1
FPS: 8.1
FPS: 8.1
FPS: 8.1
FPS: 8.1
FPS: 8.1
FPS: 8.1
FPS: 8.1
FPS: 8.1
FPS: 8.1
FPS: 8.1
FPS: 8.1
FPS: 8.1
FPS: 8.1
FPS: 8.1
FPS: 8.1
FPS: 8.1
FPS: 8.1
FPS: 8.1
FPS: 8.1
FPS: 8.1
FPS: 8.1
FPS: 8.1
FPS: 8.1
FPS: 8.1
FPS: 8.1
FPS: 8.1
FPS: 8.1
FPS: 8.1
FPS: 8.1
FPS: 8.1
FPS: 8.1
FPS: 8.1
FPS: 8.1
FPS: 8.1
FPS: 8.1
FPS: 8.1
FPS: 8.1
FPS: 8.1
FPS: 8.1
FPS: 8.1
FPS: 8.1
FPS: 8.1
FPS: 8.1
FPS: 8.1
FPS: 8.1
FPS: 8.1
FPS: 8.1
FPS: 8.1
FPS: 8.1
FPS: 8.1
FPS: 8.1
FPS: 8.1
FPS: 8.1
FPS: 8.1
FPS: 8.1
FPS: 8.1
FPS: 8.1
FPS: 8.1
FPS: 8.1
FPS: 8.1
FPS: 8.1
FPS: 8.1
FPS: 8.1
FPS: 8.1
FPS: 8.1
FPS: 8